# 04 · Rotas acessíveis para cadeira de rodas

**Rota Livre** · a IA recomenda, para quem usa **cadeira de rodas ou tem mobilidade reduzida**, o caminho a pé pela calçada:

1. **Exclui rotas bloqueadas:** escada, travessia sem rampa (guia rebaixada), inclinação acima de **8,33%** (limite de rampa da NBR 9050) ou passagem com menos de **0,90 m**.
2. Entre as acessíveis, escolhe a de **menor índice** = (buracos + obstáculos) a cada 100 m.

> ⚠️ **Dados simulados.** As ruas existem, mas as condições das calçadas (`base/calcadas_simuladas.json`) foram geradas para o MVP.
> Em produção, viriam dos relatos do app, do SP156 e de vistorias.

Mesmas 3 configurações do projeto: **sem contexto** (LLM sozinha), **RAG** (busca os textos das rotas e a LLM decide sozinha)
e **agente** (identifica origem/destino, usa a **ferramenta** que exclui as rotas bloqueadas e ordena as acessíveis; a LLM só explica, com validação e recusa).

In [ ]:
import sys, json
sys.path.append("..")
import pandas as pd
from src import rotas, llm
print("LLM local disponível?", llm.disponivel())
print(len(rotas.TRECHOS), "trechos de calçada ·", len(rotas.ROTAS), "rotas ·", len(rotas.LOCAIS), "locais")

## A base: rotas candidatas por trajeto

In [ ]:
linhas = []
for r in sorted(rotas.ROTAS, key=lambda r: (r["origem"], r["destino"], not r["acessivel"], r["indice"])):
    linhas.append({"trajeto": f"{rotas.nome_local(r['origem'])} → {rotas.nome_local(r['destino'])}", "rota": r["id"],
                   "por": " → ".join(rotas.nomes_trechos(r)), "metros": r["metros"],
                   "buracos": r["buracos"], "obstáculos": r["obstaculos"], "índice/100m": r["indice"],
                   "acessível": "sim" if r["acessivel"] else "NÃO: " + "; ".join(r["bloqueios"])})
pd.set_option("display.max_colwidth", 90)
pd.DataFrame(linhas)

Repare que, em **4 dos 6 trajetos, o caminho mais curto é intransitável** para cadeira de rodas — e, de Consolação ao MASP, a rota com **menos buracos** tem uma travessia sem rampa. É exatamente a informação que falta a quem tem mobilidade reduzida.

## Exemplo: a mesma pergunta nas 3 configurações

In [ ]:
pergunta = "Qual a melhor rota em cadeira de rodas do Metrô Sé até o Theatro Municipal?"
for modo in ["sem_contexto", "rag", "agente"]:
    r = rotas.responder(pergunta, modo)
    print(f"\n===== {modo} =====")
    print(json.dumps(r["json"], ensure_ascii=False, indent=2))
    for p in r["passos"]:
        print("  ·", p)

## Trajeto fora da base (esperado: "não consta")

In [ ]:
pergunta = "Qual o caminho acessível do Shopping Eldorado até a Estação da Luz?"
for modo in ["sem_contexto", "rag", "agente"]:
    r = rotas.responder(pergunta, modo)
    print(f"{modo:>12}:", json.dumps(r["json"], ensure_ascii=False))

## Placar (12 casos: 8 trajetos da base + 4 fora da base)

A métrica mais importante é **"recomendou rota bloqueada"**: mandar um cadeirante para uma escada é o pior erro possível.

In [ ]:
resultados = rotas.avaliar()   # salva avaliacao/rotas_resultados.json e avaliacao/rotas_placar.md
pd.DataFrame([r["metricas"] for r in resultados.values()])

## Erros de cada configuração

In [ ]:
for nome, r in resultados.items():
    erros = [e for e in r["execucoes"] if not e["correto"]]
    print(f"\n===== {nome}: {len(erros)} erros =====")
    for e in erros:
        print(f"- {e['caso_id']} ({e['tipo']}): {json.dumps(e['json'], ensure_ascii=False)[:150]}")

### Interpretação

- **Sem contexto:** sem os dados das calçadas, o modelo não sabe onde há escadas, rampas ou buracos — recusa ou inventa ruas.
- **RAG:** recebe os dados, mas precisa **cruzar bloqueios e números** sozinho; um modelo de 3B pode escolher a rota mais curta ou com menos buracos mesmo quando ela tem escada ou travessia sem rampa.
- **Agente:** a regra de acessibilidade é aplicada por **código** (a rota bloqueada nunca é recomendada) e a LLM só explica; trajetos fora da base são recusados.

**Conclusão:** quando um erro pode deixar uma pessoa em cadeira de rodas diante de uma escada, a decisão não pode ficar só com a LLM — o agente com ferramenta é a configuração do MVP.